<a href="https://colab.research.google.com/github/jeffheaton/app_generative_ai/blob/main/t81_559_class_02_2_multi_prompt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T81-559: Applications of Generative Artificial Intelligence
**Module 2: Code Generation**
* Instructor: [Jeff Heaton](https://sites.wustl.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.wustl.edu/Programs/Pages/default.aspx)
* For more information visit the [class website](https://sites.wustl.edu/jeffheaton/t81-558/).

# Module 2 Material

* Part 2.1: Prompting for Code Generation [[Video]](https://www.youtube.com/watch?v=HVId6kYKKgQ) [[Notebook]](t81_559_class_02_1_dev.ipynb)
* **Part 2.2: Handling Revision Prompts** [[Video]](https://www.youtube.com/watch?v=APpV46tplXA) [[Notebook]](t81_559_class_02_2_multi_prompt.ipynb)
* Part 2.3: Using a LLM to Help Debug [[Video]](https://www.youtube.com/watch?v=VPqSNb38QK0) [[Notebook]](t81_559_class_02_3_llm_debug.ipynb)
* Part 2.4: Tracking Prompts in Software Development [[Video]](https://www.youtube.com/watch?v=oUFUuYfvXZU) [[Notebook]](t81_559_class_02_4_software_eng.ipynb)
* Part 2.5: Limits of LLM Code Generation [[Video]](https://www.youtube.com/watch?v=dKtRI0LZSyY) [[Notebook]](t81_559_class_02_5_code_gen_limits.ipynb)


# Google CoLab Instructions

The following code ensures that Google CoLab is running and maps Google Drive if needed.

In [1]:
import os

try:
    from google.colab import drive, userdata
    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# OpenAI Secrets
if COLAB:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Install needed libraries in CoLab
if COLAB:
    !pip install langchain langchain_openai

Note: using Google CoLab
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.9/515.9 kB 22.5 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28


# 2.2: Handling Revision Prompts

Previously, we just sent one prompt to the LLM, which generated code. It is possible to perform this code more conversationally. In this module, we will see how to converse with the LLM to request changes to outputted code and even help the LLM to produce a more accurate model.

We will also see that it might be beneficial to recreate your conversation as one single prompt that generates the final result. Keeping track of one prompt, rather than a conversation, that created your final code is more maintainable.

## Conversational Code Generation

We will introduce a more advanced code generation function that allows you to start the conversation to generate code and follow up with additional prompts if needed.

In future modules, we will see how to create chatbots similar to this one. We will use the code I provided to generate your code for now. This generator uses a system prompt that requests that the generated code conform to the following:

* Imports should be sorted
* Code should conform to PEP-8 formatting
* Do not mix uncompilable notes with code
* Add comments

In [11]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
# from langchain.chains import ConversationChain
# from langchain.memory import ConversationBufferWindowMemory
from langchain_openai import ChatOpenAI
# from langchain_core.prompts.chat import PromptTemplate
from langchain_core.runnables.history import RunnableWithMessageHistory
from IPython.display import display_markdown

MODEL = 'gpt-5-mini'

store = {}

TEMPLATE = """The following is a friendly conversation between a human and an
AI to generate Python code. If you have notes about the code, place them before
the code. Any nots about execution should follow the code. If you do mix any
notes with the code, make them comments. Add proper comments to the code.
Sort imports and follow PEP-8 formatting.

Current conversation:
{history}
Human: {input}
Code Assistant:"""
# PROMPT_TEMPLATE = PromptTemplate(input_variables=["history", "input"], template=TEMPLATE)

def get_session_history(session_id: str):
  if session_id not in store:
    store[session_id] = InMemoryChatMessageHistory()
  return store[session_id]

def start_conversation():
    # Initialize the OpenAI LLM with your API key
    llm = ChatOpenAI(
        model=MODEL,
        temperature=0.0,
        n=1
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system", TEMPLATE),
        MessagesPlaceholder(variable_name="history"),
         ("human", "{input}"),
        ])

    chain = prompt | llm

    # Initialize memory and conversation
    # memory = ConversationBufferWindowMemory()
    # conversation = ConversationChain(
    #     prompt=PROMPT_TEMPLATE,
    #     llm=llm,
    #     memory=memory,
    #     verbose=False
    # )

    conversation = RunnableWithMessageHistory(
        chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="history",
    )

    return conversation

def generate_code(conversation, prompt, session_id='default'):
    print("Model response:")
    # output = conversation.invoke(prompt)
    output = conversation.invoke(
        {"input": prompt},
        config={"configurable": {"session_id": session_id}},
    )
    # display_markdown(output['response'], raw=True)
    display_markdown(output.content, raw=True)


## First Attempt at an XOR Approximator

We will construct a prompt that requests the LLM to generate a PyTorch neural network to approximate the [Exclusive Or](https://en.wikipedia.org/wiki/Exclusive_or). The truth table for the Exclusive Or (XOR) function is provided here:

```
0 XOR 0 = 0
1 XOR 0 = 1
0 XOR 1 = 1
1 XOR 1 = 0
```

If given data, neural networks can learn to approximate functions, so let's create a PyTorch neural network to approximate the XOR function.

In [12]:
conversation = start_conversation()
generate_code(conversation, """Write Python code to learn the XOR function with PyTorch.""")

Model response:


Note: This script trains a small feedforward neural network (one hidden
layer) to learn the XOR function using PyTorch. Requires PyTorch installed.

```python
# XOR with PyTorch
# This script trains a small MLP (2 -> 2 -> 1) to learn the XOR truth table.
# Uses BCEWithLogitsLoss (stable combination of sigmoid + BCE).
# Save as xor_pytorch.py and run: python xor_pytorch.py

import torch
import torch.nn as nn
import torch.optim as optim


# Reproducibility: set random seed
torch.manual_seed(0)


class XORNet(nn.Module):
    """Simple 2-2-1 feedforward network for XOR."""

    def __init__(self):
        super().__init__()
        # Hidden layer with 2 units
        self.hidden = nn.Linear(2, 2)
        # Output layer producing a single logit
        self.output = nn.Linear(2, 1)

    def forward(self, x):
        # Use tanh activation in the hidden layer (works well for XOR).
        x = torch.tanh(self.hidden(x))
        # Output is a logit; we'll use BCEWithLogitsLoss which applies sigmoid
        x = self.output(x)
        return x


def main():
    # XOR inputs and corresponding labels
    inputs = torch.tensor(
        [
            [0.0, 0.0],
            [0.0, 1.0],
            [1.0, 0.0],
            [1.0, 1.0],
        ],
        dtype=torch.float32,
    )
    labels = torch.tensor([[0.0], [1.0], [1.0], [0.0]], dtype=torch.float32)

    # Instantiate model, loss, and optimizer
    model = XORNet()
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.1)

    # Training loop
    epochs = 5000
    for epoch in range(1, epochs + 1):
        model.train()
        # Forward pass: compute logits
        logits = model(inputs)
        # Compute loss (BCEWithLogitsLoss expects logits)
        loss = criterion(logits, labels)

        # Backprop and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Print progress occasionally
        if epoch % 500 == 0 or epoch == 1:
            with torch.no_grad():
                # Compute predicted probabilities and binary predictions
                probs = torch.sigmoid(logits)
                preds = (probs >= 0.5).float()
                acc = (preds == labels).float().mean().item()
            print(
                f"Epoch {epoch:4d} | Loss: {loss.item():.6f} | "
                f"Accuracy: {acc * 100:.1f}%"
            )

    # Evaluation: show final outputs
    model.eval()
    with torch.no_grad():
        logits = model(inputs)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).int()

        print("\nInput -> Prob -> Predicted")
        for inp, p, pr in zip(inputs, probs, preds):
            print(f"{list(map(int, inp.tolist()))} -> {p.item():.4f} -> {pr.item()}")

    # Optional: print final weights for inspection
    print("\nFinal model parameters:")
    for name, param in model.named_parameters():
        print(f"{name}: {param.data.numpy()}")


if __name__ == "__main__":
    main()
```

Execution notes:
- Install PyTorch if you don't have it: pip install torch (or follow
  instructions at https://pytorch.org for OS/CUDA specifics).
- Run the script. You should see the loss decrease and final predictions
  matching the XOR truth table (probabilities near 0 or 1).

In [13]:
# XOR with PyTorch
# This script trains a small MLP (2 -> 2 -> 1) to learn the XOR truth table.
# Uses BCEWithLogitsLoss (stable combination of sigmoid + BCE).
# Save as xor_pytorch.py and run: python xor_pytorch.py

import torch
import torch.nn as nn
import torch.optim as optim

# Reproducibility: set random seed
torch.manual_seed(0)

class XORNet(nn.Module):
    """Simple 2-2-1 feedforward network for XOR."""

    def __init__(self):
        super().__init__()
        # Hidden layer with 2 units
        self.hidden = nn.Linear(2, 2)
        # Output layer producing a single logit
        self.output = nn.Linear(2, 1)

    def forward(self, x):
        # Use tanh activation in the hidden layer (works well for XOR).
        x = torch.tanh(self.hidden(x))
        # Output is a logit; we'll use BCEWithLogitsLoss which applies sigmoid
        x = self.output(x)
        return x


def main():
    # XOR inputs and corresponding labels
    inputs = torch.tensor(
        [
            [0.0, 0.0],
            [0.0, 1.0],
            [1.0, 0.0],
            [1.0, 1.0],
        ],
        dtype=torch.float32,
    )
    labels = torch.tensor([[0.0], [1.0], [1.0], [0.0]], dtype=torch.float32)

    # Instantiate model, loss, and optimizer
    model = XORNet()
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.1)

    # Training loop
    epochs = 5000
    for epoch in range(1, epochs + 1):
        model.train()
        # Forward pass: compute logits
        logits = model(inputs)
        # Compute loss (BCEWithLogitsLoss expects logits)
        loss = criterion(logits, labels)

        # Backprop and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Print progress occasionally
        if epoch % 500 == 0 or epoch == 1:
            with torch.no_grad():
                # Compute predicted probabilities and binary predictions
                probs = torch.sigmoid(logits)
                preds = (probs >= 0.5).float()
                acc = (preds == labels).float().mean().item()
            print(
                f"Epoch {epoch:4d} | Loss: {loss.item():.6f} | "
                f"Accuracy: {acc * 100:.1f}%"
            )

    # Evaluation: show final outputs
    model.eval()
    with torch.no_grad():
        logits = model(inputs)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).int()

        print("\nInput -> Prob -> Predicted")
        for inp, p, pr in zip(inputs, probs, preds):
            print(f"{list(map(int, inp.tolist()))} -> {p.item():.4f} -> {pr.item()}")

    # Optional: print final weights for inspection
    print("\nFinal model parameters:")
    for name, param in model.named_parameters():
        print(f"{name}: {param.data.numpy()}")


if __name__ == "__main__":
    main()

Epoch    1 | Loss: 0.715159 | Accuracy: 25.0%
Epoch  500 | Loss: 0.000853 | Accuracy: 100.0%
Epoch 1000 | Loss: 0.000286 | Accuracy: 100.0%
Epoch 1500 | Loss: 0.000144 | Accuracy: 100.0%
Epoch 2000 | Loss: 0.000085 | Accuracy: 100.0%
Epoch 2500 | Loss: 0.000055 | Accuracy: 100.0%
Epoch 3000 | Loss: 0.000038 | Accuracy: 100.0%
Epoch 3500 | Loss: 0.000027 | Accuracy: 100.0%
Epoch 4000 | Loss: 0.000019 | Accuracy: 100.0%
Epoch 4500 | Loss: 0.000014 | Accuracy: 100.0%
Epoch 5000 | Loss: 0.000011 | Accuracy: 100.0%

Input -> Prob -> Predicted
[0, 0] -> 0.0000 -> 0
[0, 1] -> 1.0000 -> 1
[1, 0] -> 1.0000 -> 1
[1, 1] -> 0.0000 -> 0

Final model parameters:
hidden.weight: [[-5.6645665  5.5749507]
 [ 5.143209  -5.322044 ]]
hidden.bias: [-2.7455046 -2.4506657]
output.weight: [[11.714736 11.630122]]
output.bias: [11.247592]


# Requesting a Change to Generated Code

If you've taken my other course, you will know I prefer PyTorch sequences over extending the nn.Module class, at least for simple neural networks like an XOR approximator. LLMs do not share this opinion. However, the LLM will gladly humor me and generate a sequence. Here, I provide an additional prompt to request this rather than resubmitting a modified version of my first prompt.

In [14]:
generate_code(conversation, """Could you make use of a PyTorch sequence rather than a nn.Module class?""")

Model response:


Note: This script trains a small feedforward network (2 -> 2 -> 1) to learn the XOR
truth table using a PyTorch nn.Sequential model instead of defining a custom
nn.Module subclass. It uses BCEWithLogitsLoss (logits + stable BCE) and Adam.

```python
# XOR using nn.Sequential in PyTorch
# Save as xor_sequential.py and run: python xor_sequential.py

import torch
import torch.nn as nn
import torch.optim as optim


# Reproducibility: set random seed
torch.manual_seed(0)


def main():
    # XOR inputs and labels
    inputs = torch.tensor(
        [
            [0.0, 0.0],
            [0.0, 1.0],
            [1.0, 0.0],
            [1.0, 1.0],
        ],
        dtype=torch.float32,
    )
    labels = torch.tensor([[0.0], [1.0], [1.0], [0.0]], dtype=torch.float32)

    # Build model with nn.Sequential (2 -> 2 -> 1)
    # Use Tanh in the hidden layer which works well for XOR.
    model = nn.Sequential(
        nn.Linear(2, 2),
        nn.Tanh(),
        nn.Linear(2, 1),  # outputs a single logit
    )

    # Loss and optimizer
    criterion = nn.BCEWithLogitsLoss()  # expects logits
    optimizer = optim.Adam(model.parameters(), lr=0.1)

    # Training loop
    epochs = 5000
    for epoch in range(1, epochs + 1):
        model.train()
        logits = model(inputs)  # forward pass (logits)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Print progress occasionally
        if epoch == 1 or epoch % 500 == 0:
            with torch.no_grad():
                probs = torch.sigmoid(logits)
                preds = (probs >= 0.5).float()
                acc = (preds == labels).float().mean().item()
            print(
                f"Epoch {epoch:4d} | Loss: {loss.item():.6f} | "
                f"Accuracy: {acc * 100:.1f}%"
            )

    # Evaluation: show final outputs
    model.eval()
    with torch.no_grad():
        logits = model(inputs)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).int()

        print("\nInput -> Prob -> Predicted")
        for inp, p, pr in zip(inputs, probs, preds):
            print(f"{list(map(int, inp.tolist()))} -> {p.item():.4f} -> {pr.item()}")

    # Print final model parameters for inspection
    print("\nFinal model parameters:")
    for name, param in model.named_parameters():
        print(f"{name}: {param.data.numpy()}")


if __name__ == "__main__":
    main()
```

Execution notes:
- Install PyTorch if needed: pip install torch (or follow instructions at https://pytorch.org for OS/CUDA specifics).
- Run the script: python xor_sequential.py
- Expect the loss to decrease and final predictions to match the XOR truth table (probabilities near 0 or 1).

# Testing the Generated Code

LLMs are not overachievers; they will implement the code you ask for and not provide much more. When we run the XOR approximator's first version, the results are only sometimes accurate, especially if we run the program multiple times.

In [15]:
# XOR using nn.Sequential in PyTorch
# Save as xor_sequential.py and run: python xor_sequential.py

import torch
import torch.nn as nn
import torch.optim as optim


# Reproducibility: set random seed
torch.manual_seed(0)


def main():
    # XOR inputs and labels
    inputs = torch.tensor(
        [
            [0.0, 0.0],
            [0.0, 1.0],
            [1.0, 0.0],
            [1.0, 1.0],
        ],
        dtype=torch.float32,
    )
    labels = torch.tensor([[0.0], [1.0], [1.0], [0.0]], dtype=torch.float32)

    # Build model with nn.Sequential (2 -> 2 -> 1)
    # Use Tanh in the hidden layer which works well for XOR.
    model = nn.Sequential(
        nn.Linear(2, 2),
        nn.Tanh(),
        nn.Linear(2, 1),  # outputs a single logit
    )

    # Loss and optimizer
    criterion = nn.BCEWithLogitsLoss()  # expects logits
    optimizer = optim.Adam(model.parameters(), lr=0.1)

    # Training loop
    epochs = 5000
    for epoch in range(1, epochs + 1):
        model.train()
        logits = model(inputs)  # forward pass (logits)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Print progress occasionally
        if epoch == 1 or epoch % 500 == 0:
            with torch.no_grad():
                probs = torch.sigmoid(logits)
                preds = (probs >= 0.5).float()
                acc = (preds == labels).float().mean().item()
            print(
                f"Epoch {epoch:4d} | Loss: {loss.item():.6f} | "
                f"Accuracy: {acc * 100:.1f}%"
            )

    # Evaluation: show final outputs
    model.eval()
    with torch.no_grad():
        logits = model(inputs)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).int()

        print("\nInput -> Prob -> Predicted")
        for inp, p, pr in zip(inputs, probs, preds):
            print(f"{list(map(int, inp.tolist()))} -> {p.item():.4f} -> {pr.item()}")

    # Print final model parameters for inspection
    print("\nFinal model parameters:")
    for name, param in model.named_parameters():
        print(f"{name}: {param.data.numpy()}")


if __name__ == "__main__":
    main()

Epoch    1 | Loss: 0.715159 | Accuracy: 25.0%
Epoch  500 | Loss: 0.000853 | Accuracy: 100.0%
Epoch 1000 | Loss: 0.000286 | Accuracy: 100.0%
Epoch 1500 | Loss: 0.000144 | Accuracy: 100.0%
Epoch 2000 | Loss: 0.000085 | Accuracy: 100.0%
Epoch 2500 | Loss: 0.000055 | Accuracy: 100.0%
Epoch 3000 | Loss: 0.000038 | Accuracy: 100.0%
Epoch 3500 | Loss: 0.000027 | Accuracy: 100.0%
Epoch 4000 | Loss: 0.000019 | Accuracy: 100.0%
Epoch 4500 | Loss: 0.000014 | Accuracy: 100.0%
Epoch 5000 | Loss: 0.000011 | Accuracy: 100.0%

Input -> Prob -> Predicted
[0, 0] -> 0.0000 -> 0
[0, 1] -> 1.0000 -> 1
[1, 0] -> 1.0000 -> 1
[1, 1] -> 0.0000 -> 0

Final model parameters:
0.weight: [[-5.6645665  5.5749507]
 [ 5.143209  -5.322044 ]]
0.bias: [-2.7455046 -2.4506657]
2.weight: [[11.714736 11.630122]]
2.bias: [11.247592]


If you receive an error or the output is not exactly what you like, it is effective to provide that output and any errors to the LLM. Here, we provide the output and ask the LLM if that seems correct. Sometimes, the LLM may insist that the output is correct, so you must "debate" the LLM, providing additional details.

In [16]:
generate_code(conversation, """The output was:

Input -> Prob -> Predicted
[0, 0] -> 0.0000 -> 0
[0, 1] -> 1.0000 -> 1
[1, 0] -> 1.0000 -> 1
[1, 1] -> 0.0000 -> 0

Final model parameters:
0.weight: [[-5.6645665  5.5749507]
 [ 5.143209  -5.322044 ]]
0.bias: [-2.7455046 -2.4506657]
2.weight: [[11.714736 11.630122]]
2.bias: [11.247592]

Are you sure that is correct?
""")

Model response:


Short answer: yes — those parameters are plausible and the model output is correct. In nn.Sequential the layer names are numeric (0 is the first Linear, 1 is Tanh, 2 is the second Linear). The hidden tanh units are driven to values near ±1, and the output linear layer combines those to give a large positive logit for XOR=1 cases and a large negative logit for XOR=0 cases. Many weight/bias configurations can implement XOR; the one you printed is a valid solution.

Below is a compact script using nn.Sequential that trains XOR, prints the named parameters (so you can see 0.weight / 0.bias and 2.weight / 2.bias), shows predictions, and also prints the step-by-step hidden activations and output logit to demonstrate why the network produces the correct outputs.

Notes about the code:
- Uses nn.Sequential (Linear -> Tanh -> Linear).
- Uses BCEWithLogitsLoss (so the network produces logits).
- Prints both model.named_parameters() (numeric layer keys) and a manual forward-pass breakdown.

```python
# XOR using nn.Sequential (2 -> 2 -> 1) with verification
# Save as xor_sequential_debug.py and run: python xor_sequential_debug.py

import torch
import torch.nn as nn
import torch.optim as optim


# Reproducibility
torch.manual_seed(0)


def main():
    # XOR inputs and targets
    inputs = torch.tensor(
        [[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]],
        dtype=torch.float32,
    )
    targets = torch.tensor([[0.0], [1.0], [1.0], [0.0]], dtype=torch.float32)

    # Build model: nn.Sequential indices -> 0: Linear(2,2), 1: Tanh(), 2: Linear(2,1)
    model = nn.Sequential(
        nn.Linear(2, 2),
        nn.Tanh(),
        nn.Linear(2, 1),
    )

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.1)

    # Train briefly (5000 epochs is fine; you can reduce if desired)
    epochs = 5000
    for epoch in range(1, epochs + 1):
        logits = model(inputs)
        loss = criterion(logits, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch == 1 or epoch % 500 == 0:
            with torch.no_grad():
                probs = torch.sigmoid(logits)
                preds = (probs >= 0.5).float()
                acc = (preds == targets).float().mean().item()
            print(f"Epoch {epoch:4d} | Loss: {loss.item():.6f} | Accuracy: {acc * 100:.1f}%")

    # Print final predictions
    model.eval()
    with torch.no_grad():
        logits = model(inputs)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).int()

        print("\nInput -> Prob -> Predicted")
        for inp, p, pr in zip(inputs, probs, preds):
            print(f"{list(map(int, inp.tolist()))} -> {p.item():.4f} -> {pr.item()}")

    # Print named parameters (numeric keys correspond to sequential indices)
    print("\nNamed parameters (layer index.name):")
    for name, param in model.named_parameters():
        print(f"{name}: {param.data.numpy()}")

    # Manual forward-pass breakdown using the same parameters to show why it works
    w1 = model[0].weight.data  # shape (2,2)
    b1 = model[0].bias.data    # shape (2,)
    w2 = model[2].weight.data  # shape (1,2)
    b2 = model[2].bias.data    # shape (1,)

    print("\nManual forward-pass breakdown (pre-activation, tanh, output logit):")
    for x in inputs:
        # pre-activation for hidden layer: w1 @ x + b1
        pre = w1.matmul(x) + b1
        h = torch.tanh(pre)
        logit = w2.matmul(h) + b2  # shape (1,)
        print(
            f"input={list(map(int, x.tolist()))} | pre={pre.numpy()} | "
            f"h={h.numpy()} | logit={logit.item():.6f}"
        )


if __name__ == "__main__":
    main()
```

Execution notes:
- Install PyTorch if needed (pip install torch) or follow instructions at https://pytorch.org.
- Run the script. It will show the numeric-layer parameter names (0.weight, 0.bias, 2.weight, 2.bias) and a manual breakdown demonstrating the tanh hidden outputs are near ±1 so the output linear layer yields large positive/negative logits that give the correct XOR behavior.

If you want, I can also show a short algebraic explanation of the particular numbers you printed (walk through the four inputs numerically to confirm the signs of the pre-activations and final logits).

## Test the Improved Version

We now receive much more accurate output when we test the neural network provided.

In [17]:
import torch
import torch.nn as nn
import torch.optim as optim


# Reproducibility
torch.manual_seed(0)


def main():
    # XOR inputs and targets
    inputs = torch.tensor(
        [[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]],
        dtype=torch.float32,
    )
    targets = torch.tensor([[0.0], [1.0], [1.0], [0.0]], dtype=torch.float32)

    # Build model: nn.Sequential indices -> 0: Linear(2,2), 1: Tanh(), 2: Linear(2,1)
    model = nn.Sequential(
        nn.Linear(2, 2),
        nn.Tanh(),
        nn.Linear(2, 1),
    )

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.1)

    # Train briefly (5000 epochs is fine; you can reduce if desired)
    epochs = 5000
    for epoch in range(1, epochs + 1):
        logits = model(inputs)
        loss = criterion(logits, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch == 1 or epoch % 500 == 0:
            with torch.no_grad():
                probs = torch.sigmoid(logits)
                preds = (probs >= 0.5).float()
                acc = (preds == targets).float().mean().item()
            print(f"Epoch {epoch:4d} | Loss: {loss.item():.6f} | Accuracy: {acc * 100:.1f}%")

    # Print final predictions
    model.eval()
    with torch.no_grad():
        logits = model(inputs)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).int()

        print("\nInput -> Prob -> Predicted")
        for inp, p, pr in zip(inputs, probs, preds):
            print(f"{list(map(int, inp.tolist()))} -> {p.item():.4f} -> {pr.item()}")

    # Print named parameters (numeric keys correspond to sequential indices)
    print("\nNamed parameters (layer index.name):")
    for name, param in model.named_parameters():
        print(f"{name}: {param.data.numpy()}")

    # Manual forward-pass breakdown using the same parameters to show why it works
    w1 = model[0].weight.data  # shape (2,2)
    b1 = model[0].bias.data    # shape (2,)
    w2 = model[2].weight.data  # shape (1,2)
    b2 = model[2].bias.data    # shape (1,)

    print("\nManual forward-pass breakdown (pre-activation, tanh, output logit):")
    for x in inputs:
        # pre-activation for hidden layer: w1 @ x + b1
        pre = w1.matmul(x) + b1
        h = torch.tanh(pre)
        logit = w2.matmul(h) + b2  # shape (1,)
        print(
            f"input={list(map(int, x.tolist()))} | pre={pre.numpy()} | "
            f"h={h.numpy()} | logit={logit.item():.6f}"
        )


if __name__ == "__main__":
    main()

Epoch    1 | Loss: 0.715159 | Accuracy: 25.0%
Epoch  500 | Loss: 0.000853 | Accuracy: 100.0%
Epoch 1000 | Loss: 0.000286 | Accuracy: 100.0%
Epoch 1500 | Loss: 0.000144 | Accuracy: 100.0%
Epoch 2000 | Loss: 0.000085 | Accuracy: 100.0%
Epoch 2500 | Loss: 0.000055 | Accuracy: 100.0%
Epoch 3000 | Loss: 0.000038 | Accuracy: 100.0%
Epoch 3500 | Loss: 0.000027 | Accuracy: 100.0%
Epoch 4000 | Loss: 0.000019 | Accuracy: 100.0%
Epoch 4500 | Loss: 0.000014 | Accuracy: 100.0%
Epoch 5000 | Loss: 0.000011 | Accuracy: 100.0%

Input -> Prob -> Predicted
[0, 0] -> 0.0000 -> 0
[0, 1] -> 1.0000 -> 1
[1, 0] -> 1.0000 -> 1
[1, 1] -> 0.0000 -> 0

Named parameters (layer index.name):
0.weight: [[-5.6645665  5.5749507]
 [ 5.143209  -5.322044 ]]
0.bias: [-2.7455046 -2.4506657]
2.weight: [[11.714736 11.630122]]
2.bias: [11.247592]

Manual forward-pass breakdown (pre-activation, tanh, output logit):
input=[0, 0] | pre=[-2.7455046 -2.4506657] | h=[-0.9917865  -0.98523647] | logit=-11.829347
input=[0, 1] | pre=[ 2

## Combining the Conversation into a Single Prompt

We should combine this entire conversation into a single prompt, especially if we wish to save the prompt along with the code. We can request the LLM to create this combined prompt for us.

In [18]:
generate_code(conversation, """Okay, that is great, can you suggest a single
prompt that would have resulted in this last code output?""")

Model response:


Write a complete, runnable Python script that trains a small feedforward network to learn the XOR truth table using PyTorch's nn.Sequential (Linear(2,2) -> Tanh -> Linear(2,1)). Requirements:
- Use torch.manual_seed(0) for reproducibility.
- Use BCEWithLogitsLoss and optim.Adam with lr=0.1.
- Train for 5000 epochs and print progress at epoch 1 and every 500 epochs (loss and accuracy).
- After training, print the four inputs with predicted probability and binary prediction (Input -> Prob -> Predicted).
- Print model.named_parameters() so the layer names are numeric (e.g., 0.weight, 0.bias, 2.weight, 2.bias).
- Print a manual forward-pass breakdown for each input showing hidden pre-activation, tanh(hidden), and final output logit numeric values.
- Include clear comments, follow PEP 8, sort imports, and include a main guard so the script can be run directly.

The LLM's attempt at a consoldated prompt is incomplete. It skips several important details and does not provide precise requirements. I will manually make some improvements, which you can see here.

In [21]:
# Start a new conversation
conversation = start_conversation()
generate_code(conversation, """
Write a complete, runnable Python script that trains a small feedforward network to learn the XOR truth table using PyTorch's nn.Sequential (Linear(2,2) -> Tanh -> Linear(2,1)).
Requirements:
Use torch.manual_seed(0) for reproducibility.
Use BCEWithLogitsLoss and optim.Adam with lr=0.1.
Train for 5000 epochs and print progress at epoch 1 and every 500 epochs (loss and accuracy).
After training, print the four inputs with predicted probability and binary prediction (Input -> Prob -> Predicted).
Print model.named_parameters() so the layer names are numeric (e.g., 0.weight, 0.bias, 2.weight, 2.bias).
Print a manual forward-pass breakdown for each input showing hidden pre-activation, tanh(hidden), and final output logit numeric values.
Include clear comments, follow PEP 8, sort imports, and include a main guard so the script can be run directly.""")

Model response:


Note: The following script trains an nn.Sequential network (Linear(2,2) -> Tanh -> Linear(2,1))
to learn the XOR truth table. It uses BCEWithLogitsLoss, optim.Adam with lr=0.1,
torch.manual_seed(0), trains for 5000 epochs, and prints progress, final
predictions, named parameters (numeric layer names), and a manual forward-pass
breakdown for each input.

```python
# xor_sequential_2hidden.py
# Train nn.Sequential (2 -> 2 -> 1) to learn XOR using PyTorch.
# - Uses torch.manual_seed(0) for reproducibility.
# - Uses BCEWithLogitsLoss and Adam (lr=0.1).
# - Trains for 5000 epochs, prints at epoch 1 and every 500 epochs.
# - Prints final predictions, named_parameters, and manual forward-pass values.
# Save as xor_sequential_2hidden.py and run: python xor_sequential_2hidden.py

import torch
import torch.nn as nn
import torch.optim as optim


# Reproducibility
torch.manual_seed(0)


def main():
    # XOR inputs and corresponding labels (4 examples)
    inputs = torch.tensor(
        [[0.0, 0.0],
         [0.0, 1.0],
         [1.0, 0.0],
         [1.0, 1.0]],
        dtype=torch.float32,
    )
    labels = torch.tensor([[0.0], [1.0], [1.0], [0.0]], dtype=torch.float32)

    # Build model using nn.Sequential: Linear(2,2) -> Tanh -> Linear(2,1)
    model = nn.Sequential(
        nn.Linear(2, 2),
        nn.Tanh(),
        nn.Linear(2, 1),  # outputs a single logit
    )

    # Loss and optimizer
    criterion = nn.BCEWithLogitsLoss()  # expects logits
    optimizer = optim.Adam(model.parameters(), lr=0.1)

    # Training settings
    epochs = 5000

    # Training loop (full-batch)
    for epoch in range(1, epochs + 1):
        model.train()
        logits = model(inputs)  # forward pass (logits)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Print progress at epoch 1 and every 500 epochs
        if epoch == 1 or epoch % 500 == 0:
            with torch.no_grad():
                probs = torch.sigmoid(logits)
                preds = (probs >= 0.5).float()
                acc = (preds == labels).float().mean().item()
            print(
                f"Epoch {epoch:4d} | Loss: {loss.item():.6f} | "
                f"Accuracy: {acc * 100:.1f}%"
            )

    # Evaluation: final predictions
    model.eval()
    with torch.no_grad():
        logits = model(inputs)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).int()

        print("\nInput -> Prob -> Predicted")
        for inp, p, pr in zip(inputs, probs, preds):
            print(f"{list(map(int, inp.tolist()))} -> {p.item():.4f} -> {pr.item()}")

    # Print named parameters so layer names are numeric (0, 1, 2)
    print("\nNamed parameters (layer_index.parameter):")
    for name, param in model.named_parameters():
        # Convert parameter tensor to numpy for compact display
        print(f"{name}: {param.data.numpy()}")

    # Manual forward-pass breakdown for each input:
    # show hidden pre-activation, tanh(hidden), and final output logit values
    w1 = model[0].weight.data  # shape (2,2)
    b1 = model[0].bias.data    # shape (2,)
    w2 = model[2].weight.data  # shape (1,2)
    b2 = model[2].bias.data    # shape (1,)

    print("\nManual forward-pass breakdown (hidden pre-activation, tanh(hidden), output logit):")
    for x in inputs:
        pre = w1.matmul(x) + b1      # pre-activation of hidden (shape (2,))
        h = torch.tanh(pre)          # hidden activation (shape (2,))
        logit = w2.matmul(h) + b2    # output logit (shape (1,))
        # Print numeric arrays for inspection
        print(
            f"input={list(map(int, x.tolist()))} | "
            f"pre={pre.numpy()} | h={h.numpy()} | logit={logit.item():.6f}"
        )


if __name__ == "__main__":
    main()
```

Execution notes:
- Requires PyTorch. Install with: pip install torch (or follow instructions at https://pytorch.org).
- Run the script: python xor_sequential_2hidden.py
- You should see the loss printed at epoch 1 and every 500 epochs, the final
  predicted probabilities and binary predictions, the numeric named parameters
  (0.weight, 0.bias, 2.weight, 2.bias), and a manual breakdown showing how the
  tanh hidden activations produce the final logits.

In [19]:
# Start a new conversation
conversation = start_conversation()
generate_code(conversation, """
Can you provide Python code using PyTorch to effectively learn the XOR function
with 4 hidden neurons, using the Adam optimizer, and 20K training epochs?
Use a sequence not a nn.Module class.""")

Model response:


Note: The following script trains an nn.Sequential feedforward network (2 -> 4 -> 1)
to learn XOR using PyTorch. It uses BCEWithLogitsLoss, Adam, torch.manual_seed(0),
and runs for 20,000 epochs. The script prints progress, final predictions,
named parameters, and a manual forward-pass breakdown for verification.

```python
# xor_sequential_4hidden_20k.py
# Train an nn.Sequential network (2 -> 4 -> 1) to learn XOR using PyTorch.
# Uses BCEWithLogitsLoss and Adam, trains for 20_000 epochs.
# Save and run: python xor_sequential_4hidden_20k.py

import torch
import torch.nn as nn
import torch.optim as optim


# Reproducibility
torch.manual_seed(0)


def main():
    # XOR inputs and targets (4 examples)
    inputs = torch.tensor(
        [[0.0, 0.0],
         [0.0, 1.0],
         [1.0, 0.0],
         [1.0, 1.0]],
        dtype=torch.float32,
    )
    targets = torch.tensor([[0.0], [1.0], [1.0], [0.0]], dtype=torch.float32)

    # Build model using nn.Sequential: Linear(2,4) -> Tanh -> Linear(4,1)
    model = nn.Sequential(
        nn.Linear(2, 4),
        nn.Tanh(),
        nn.Linear(4, 1),  # outputs a single logit
    )

    # Loss and optimizer
    criterion = nn.BCEWithLogitsLoss()  # expects logits
    optimizer = optim.Adam(model.parameters(), lr=0.1)

    # Training settings
    epochs = 20_000
    print_every = 2_000  # print at epoch 1 and every `print_every` epochs

    # Training loop
    for epoch in range(1, epochs + 1):
        model.train()
        logits = model(inputs)  # forward pass (logits)
        loss = criterion(logits, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Print progress occasionally
        if epoch == 1 or epoch % print_every == 0:
            with torch.no_grad():
                probs = torch.sigmoid(logits)
                preds = (probs >= 0.5).float()
                acc = (preds == targets).float().mean().item()
            print(
                f"Epoch {epoch:5d} | Loss: {loss.item():.6f} | "
                f"Accuracy: {acc * 100:.1f}%"
            )

    # Evaluation: final predictions
    model.eval()
    with torch.no_grad():
        logits = model(inputs)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).int()

        print("\nInput -> Prob -> Predicted")
        for inp, p, pr in zip(inputs, probs, preds):
            print(f"{list(map(int, inp.tolist()))} -> {p.item():.4f} -> {pr.item()}")

    # Print named parameters (numeric indices correspond to sequential layers)
    print("\nNamed parameters (layer_index.parameter):")
    for name, param in model.named_parameters():
        print(f"{name}: {param.data.numpy()}")

    # Manual forward-pass breakdown for each input to show internal values
    w1 = model[0].weight.data  # shape (4,2)
    b1 = model[0].bias.data    # shape (4,)
    w2 = model[2].weight.data  # shape (1,4)
    b2 = model[2].bias.data    # shape (1,)

    print("\nManual forward-pass breakdown (hidden pre-activation, tanh(hidden), output logit):")
    for x in inputs:
        pre = w1.matmul(x) + b1       # pre-activation of hidden (shape (4,))
        h = torch.tanh(pre)           # hidden activation (shape (4,))
        logit = w2.matmul(h) + b2     # output logit (shape (1,))
        # Convert tensors to numpy for readable printing
        print(
            f"input={list(map(int, x.tolist()))} | "
            f"pre={pre.numpy()} | h={h.numpy()} | logit={logit.item():.6f}"
        )


if __name__ == "__main__":
    main()
```

Execution notes:
- Requires PyTorch. Install with pip install torch (or follow installation instructions at https://pytorch.org).
- Training for 20,000 epochs on this tiny dataset is fast on CPU; adjust lr or epochs if you want quicker runs.
- The script uses nn.Sequential (no custom nn.Module subclass).

## Test the Final Prompt

Now, we test the final prompt. My prompt produces an acceptable result, but there are some opportunities for improvement. You can specify the exact format for the output. For example, sometimes code is generated to round the results, but other times it is not.

In [22]:
# xor_sequential_2hidden.py
# Train nn.Sequential (2 -> 2 -> 1) to learn XOR using PyTorch.
# - Uses torch.manual_seed(0) for reproducibility.
# - Uses BCEWithLogitsLoss and Adam (lr=0.1).
# - Trains for 5000 epochs, prints at epoch 1 and every 500 epochs.
# - Prints final predictions, named_parameters, and manual forward-pass values.
# Save as xor_sequential_2hidden.py and run: python xor_sequential_2hidden.py

import torch
import torch.nn as nn
import torch.optim as optim


# Reproducibility
torch.manual_seed(0)


def main():
    # XOR inputs and corresponding labels (4 examples)
    inputs = torch.tensor(
        [[0.0, 0.0],
         [0.0, 1.0],
         [1.0, 0.0],
         [1.0, 1.0]],
        dtype=torch.float32,
    )
    labels = torch.tensor([[0.0], [1.0], [1.0], [0.0]], dtype=torch.float32)

    # Build model using nn.Sequential: Linear(2,2) -> Tanh -> Linear(2,1)
    model = nn.Sequential(
        nn.Linear(2, 2),
        nn.Tanh(),
        nn.Linear(2, 1),  # outputs a single logit
    )

    # Loss and optimizer
    criterion = nn.BCEWithLogitsLoss()  # expects logits
    optimizer = optim.Adam(model.parameters(), lr=0.1)

    # Training settings
    epochs = 5000

    # Training loop (full-batch)
    for epoch in range(1, epochs + 1):
        model.train()
        logits = model(inputs)  # forward pass (logits)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Print progress at epoch 1 and every 500 epochs
        if epoch == 1 or epoch % 500 == 0:
            with torch.no_grad():
                probs = torch.sigmoid(logits)
                preds = (probs >= 0.5).float()
                acc = (preds == labels).float().mean().item()
            print(
                f"Epoch {epoch:4d} | Loss: {loss.item():.6f} | "
                f"Accuracy: {acc * 100:.1f}%"
            )

    # Evaluation: final predictions
    model.eval()
    with torch.no_grad():
        logits = model(inputs)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).int()

        print("\nInput -> Prob -> Predicted")
        for inp, p, pr in zip(inputs, probs, preds):
            print(f"{list(map(int, inp.tolist()))} -> {p.item():.4f} -> {pr.item()}")

    # Print named parameters so layer names are numeric (0, 1, 2)
    print("\nNamed parameters (layer_index.parameter):")
    for name, param in model.named_parameters():
        # Convert parameter tensor to numpy for compact display
        print(f"{name}: {param.data.numpy()}")

    # Manual forward-pass breakdown for each input:
    # show hidden pre-activation, tanh(hidden), and final output logit values
    w1 = model[0].weight.data  # shape (2,2)
    b1 = model[0].bias.data    # shape (2,)
    w2 = model[2].weight.data  # shape (1,2)
    b2 = model[2].bias.data    # shape (1,)

    print("\nManual forward-pass breakdown (hidden pre-activation, tanh(hidden), output logit):")
    for x in inputs:
        pre = w1.matmul(x) + b1      # pre-activation of hidden (shape (2,))
        h = torch.tanh(pre)          # hidden activation (shape (2,))
        logit = w2.matmul(h) + b2    # output logit (shape (1,))
        # Print numeric arrays for inspection
        print(
            f"input={list(map(int, x.tolist()))} | "
            f"pre={pre.numpy()} | h={h.numpy()} | logit={logit.item():.6f}"
        )


if __name__ == "__main__":
    main()

Epoch    1 | Loss: 0.715159 | Accuracy: 25.0%
Epoch  500 | Loss: 0.000853 | Accuracy: 100.0%
Epoch 1000 | Loss: 0.000286 | Accuracy: 100.0%
Epoch 1500 | Loss: 0.000144 | Accuracy: 100.0%
Epoch 2000 | Loss: 0.000085 | Accuracy: 100.0%
Epoch 2500 | Loss: 0.000055 | Accuracy: 100.0%
Epoch 3000 | Loss: 0.000038 | Accuracy: 100.0%
Epoch 3500 | Loss: 0.000027 | Accuracy: 100.0%
Epoch 4000 | Loss: 0.000019 | Accuracy: 100.0%
Epoch 4500 | Loss: 0.000014 | Accuracy: 100.0%
Epoch 5000 | Loss: 0.000011 | Accuracy: 100.0%

Input -> Prob -> Predicted
[0, 0] -> 0.0000 -> 0
[0, 1] -> 1.0000 -> 1
[1, 0] -> 1.0000 -> 1
[1, 1] -> 0.0000 -> 0

Named parameters (layer_index.parameter):
0.weight: [[-5.6645665  5.5749507]
 [ 5.143209  -5.322044 ]]
0.bias: [-2.7455046 -2.4506657]
2.weight: [[11.714736 11.630122]]
2.bias: [11.247592]

Manual forward-pass breakdown (hidden pre-activation, tanh(hidden), output logit):
input=[0, 0] | pre=[-2.7455046 -2.4506657] | h=[-0.9917865  -0.98523647] | logit=-11.829347
in

In [20]:
# xor_sequential_4hidden_20k.py
# Train an nn.Sequential network (2 -> 4 -> 1) to learn XOR using PyTorch.
# Uses BCEWithLogitsLoss and Adam, trains for 20_000 epochs.
# Save and run: python xor_sequential_4hidden_20k.py

import torch
import torch.nn as nn
import torch.optim as optim


# Reproducibility
torch.manual_seed(0)


def main():
    # XOR inputs and targets (4 examples)
    inputs = torch.tensor(
        [[0.0, 0.0],
         [0.0, 1.0],
         [1.0, 0.0],
         [1.0, 1.0]],
        dtype=torch.float32,
    )
    targets = torch.tensor([[0.0], [1.0], [1.0], [0.0]], dtype=torch.float32)

    # Build model using nn.Sequential: Linear(2,4) -> Tanh -> Linear(4,1)
    model = nn.Sequential(
        nn.Linear(2, 4),
        nn.Tanh(),
        nn.Linear(4, 1),  # outputs a single logit
    )

    # Loss and optimizer
    criterion = nn.BCEWithLogitsLoss()  # expects logits
    optimizer = optim.Adam(model.parameters(), lr=0.1)

    # Training settings
    epochs = 20_000
    print_every = 2_000  # print at epoch 1 and every `print_every` epochs

    # Training loop
    for epoch in range(1, epochs + 1):
        model.train()
        logits = model(inputs)  # forward pass (logits)
        loss = criterion(logits, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Print progress occasionally
        if epoch == 1 or epoch % print_every == 0:
            with torch.no_grad():
                probs = torch.sigmoid(logits)
                preds = (probs >= 0.5).float()
                acc = (preds == targets).float().mean().item()
            print(
                f"Epoch {epoch:5d} | Loss: {loss.item():.6f} | "
                f"Accuracy: {acc * 100:.1f}%"
            )

    # Evaluation: final predictions
    model.eval()
    with torch.no_grad():
        logits = model(inputs)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).int()

        print("\nInput -> Prob -> Predicted")
        for inp, p, pr in zip(inputs, probs, preds):
            print(f"{list(map(int, inp.tolist()))} -> {p.item():.4f} -> {pr.item()}")

    # Print named parameters (numeric indices correspond to sequential layers)
    print("\nNamed parameters (layer_index.parameter):")
    for name, param in model.named_parameters():
        print(f"{name}: {param.data.numpy()}")

    # Manual forward-pass breakdown for each input to show internal values
    w1 = model[0].weight.data  # shape (4,2)
    b1 = model[0].bias.data    # shape (4,)
    w2 = model[2].weight.data  # shape (1,4)
    b2 = model[2].bias.data    # shape (1,)

    print("\nManual forward-pass breakdown (hidden pre-activation, tanh(hidden), output logit):")
    for x in inputs:
        pre = w1.matmul(x) + b1       # pre-activation of hidden (shape (4,))
        h = torch.tanh(pre)           # hidden activation (shape (4,))
        logit = w2.matmul(h) + b2     # output logit (shape (1,))
        # Convert tensors to numpy for readable printing
        print(
            f"input={list(map(int, x.tolist()))} | "
            f"pre={pre.numpy()} | h={h.numpy()} | logit={logit.item():.6f}"
        )


if __name__ == "__main__":
    main()

Epoch     1 | Loss: 0.699566 | Accuracy: 50.0%
Epoch  2000 | Loss: 0.000020 | Accuracy: 100.0%
Epoch  4000 | Loss: 0.000005 | Accuracy: 100.0%
Epoch  6000 | Loss: 0.000002 | Accuracy: 100.0%
Epoch  8000 | Loss: 0.000001 | Accuracy: 100.0%
Epoch 10000 | Loss: 0.000000 | Accuracy: 100.0%
Epoch 12000 | Loss: 0.000000 | Accuracy: 100.0%
Epoch 14000 | Loss: 0.000000 | Accuracy: 100.0%
Epoch 16000 | Loss: 0.000000 | Accuracy: 100.0%
Epoch 18000 | Loss: 0.000000 | Accuracy: 100.0%
Epoch 20000 | Loss: 0.000000 | Accuracy: 100.0%

Input -> Prob -> Predicted
[0, 0] -> 0.0000 -> 0
[0, 1] -> 1.0000 -> 1
[1, 0] -> 1.0000 -> 1
[1, 1] -> 0.0000 -> 0

Named parameters (layer_index.parameter):
0.weight: [[-4.763121   4.703906 ]
 [-5.084875  -4.8086023]
 [ 5.1951275 -5.281161 ]
 [ 3.914299   4.3015575]]
0.bias: [ 2.5815897  2.4138153  3.0154703 -6.12108  ]
2.weight: [[-11.278997  -7.946781 -11.41939   -8.175558]]
2.bias: [1.2377356]

Manual forward-pass breakdown (hidden pre-activation, tanh(hidden), ou